# 찜과 채팅 사이

중고거래 서비스의 조회 - 찜 - 채팅 - 거래 퍼널을 분석하고,
찜 이후 채팅 전환에 영향을 주는 행동 조건을 탐색한다.

- 분석 기간: 2025-12-01 ~ 2026-02-28
- 분석 단위: 사용자 × 판매글
- 채팅 기준: 채팅방 생성이 아닌 첫 메시지 발생
- 데이터: MySQL에 적재된 합성 데이터


## 0. 실행 준비


In [1]:
# 라이브러리 불러오기
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 20260228
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.4f}'.format)


In [ ]:
# MySQL 연결하기
project_root = Path.cwd().resolve()
while not (project_root / '.env').exists() and project_root != project_root.parent:
    project_root = project_root.parent

load_dotenv(project_root / '.env')

db_url = URL.create(
    drivername='mysql+mysqlconnector',
    username=os.environ['MYSQL_USER'],
    password=os.environ['MYSQL_PASSWORD'] or None,
    host=os.environ['MYSQL_HOST'],
    port=int(os.environ['MYSQL_PORT']),
    database=os.environ['MYSQL_DATABASE'],
    query={'charset': 'utf8mb4'},
)

engine = create_engine(db_url, pool_pre_ping=True)
print('MySQL 연결 완료')


MySQL 연결 완료


In [3]:
# 판매글 퍼널 VIEW 불러오기
df_listing = pd.read_sql(
    'SELECT * FROM vw_listing_funnel ORDER BY listing_id',
    con=engine,
)

print('df_listing:', df_listing.shape)
df_listing.head()


df_listing: (1500, 19)


,listing_id,seller_id,category,current_price,photo_count,description_length,is_negotiable,region_id,created_at,status,seller_manner_score,seller_is_verified,view_count,unique_viewer_count,favorite_event_count,unique_favoriter_count,favorite_remove_count,chat_started_count,unique_chat_buyer_count
0,1,196,도서,"17,800.0000",1,61,1,seongnam_bundang,2026-01-16 08:55:46,예약중,39.1000,1,17,16,2,2,0.0000,3.0000,3
1,2,922,스포츠 및 레저,"961,000.0000",1,65,1,seoul_mapo,2026-02-16 22:40:53,판매중,39.1000,1,27,25,1,1,0.0000,2.0000,2
2,3,548,가구 및 인테리어,"34,600.0000",6,92,0,incheon_songdo,2025-12-27 00:10:34,판매중,31.5000,1,24,18,1,1,0.0000,1.0000,1
3,4,779,생활용품,"50,900.0000",8,97,1,seoul_gwanak,2026-01-15 14:24:58,판매중,40.1000,1,62,51,9,9,0.0000,4.0000,4
4,5,694,디지털기기,"278,000.0000",5,90,1,seoul_mapo,2025-12-31 11:09:32,판매중,37.2000,1,28,24,3,3,1.0000,2.0000,2


`df_listing`은 `vw_listing_funnel`을 불러온 데이터프레임이다.
판매글 1개가 1행이며 조회·찜·채팅 집계값과 판매글 특성을 포함한다.


## 1. 행동별 퍼널 분석


In [4]:
# 사용자별 행동 데이터 불러오기
funnel_query = '''
WITH view_pair AS (
    SELECT
        pv.user_id,
        pv.listing_id,
        COUNT(*) AS user_view_count,
        MIN(pv.viewed_at) AS first_view_at,
        MAX(pv.viewed_at) AS last_view_at
    FROM product_views pv
    JOIN listings l
        ON l.listing_id = pv.listing_id
    WHERE pv.user_id <> l.seller_id
    GROUP BY pv.user_id, pv.listing_id
),
favorite_pair AS (
    SELECT
        user_id,
        listing_id,
        COUNT(*) AS favorite_cycle_count,
        SUM(removed_at IS NOT NULL) AS favorite_remove_count,
        MAX(removed_at IS NULL) AS is_currently_favorited,
        MIN(favorited_at) AS first_favorite_at
    FROM favorites
    GROUP BY user_id, listing_id
),
chat_pair AS (
    SELECT
        buyer_id AS user_id,
        listing_id,
        MIN(created_at) AS chatroom_created_at,
        MIN(first_message_at) AS first_message_at
    FROM chatrooms
    GROUP BY buyer_id, listing_id
)
SELECT
    v.user_id,
    v.listing_id,
    v.user_view_count,
    v.first_view_at,
    v.last_view_at,
    COALESCE(f.favorite_cycle_count, 0) AS favorite_cycle_count,
    COALESCE(f.favorite_remove_count, 0) AS favorite_remove_count,
    COALESCE(f.is_currently_favorited, 0) AS is_currently_favorited,
    f.first_favorite_at,
    c.chatroom_created_at,
    c.first_message_at,
    u.region_id AS user_region_id,
    COALESCE((
        SELECT COUNT(*)
        FROM product_views pv2
        WHERE pv2.user_id = v.user_id
          AND pv2.listing_id = v.listing_id
          AND pv2.viewed_at > f.first_favorite_at
          AND (
              c.first_message_at IS NULL
              OR pv2.viewed_at <= c.first_message_at
          )
    ), 0) AS post_favorite_revisit_count
FROM view_pair v
JOIN users u
    ON u.user_id = v.user_id
LEFT JOIN favorite_pair f
    ON f.user_id = v.user_id
   AND f.listing_id = v.listing_id
LEFT JOIN chat_pair c
    ON c.user_id = v.user_id
   AND c.listing_id = v.listing_id
ORDER BY v.listing_id, v.user_id
'''

df_funnel = pd.read_sql(funnel_query, con=engine)
print('행동 원본:', df_funnel.shape)


행동 원본: (36360, 13)


In [5]:
# 시간 칼럼 변환하기
time_columns = [
    'first_view_at',
    'last_view_at',
    'first_favorite_at',
    'chatroom_created_at',
    'first_message_at',
]

df_funnel[time_columns] = df_funnel[time_columns].apply(pd.to_datetime)


In [6]:
# 판매글 특성 결합하기
listing_features = df_listing[
    [
        'listing_id',
        'category',
        'current_price',
        'photo_count',
        'description_length',
        'is_negotiable',
        'region_id',
        'created_at',
        'status',
        'seller_manner_score',
    ]
].rename(
    columns={
        'region_id': 'listing_region_id',
        'created_at': 'listing_created_at',
    }
)

df_funnel = df_funnel.merge(
    listing_features,
    on='listing_id',
    how='left',
    validate='many_to_one',
)

df_funnel['same_region'] = (
    df_funnel['user_region_id'] == df_funnel['listing_region_id']
).astype(int)

df_funnel['is_repeat_viewer'] = (
    df_funnel['user_view_count'] >= 2
).astype(int)


In [7]:
# 행동 경로 분류하기
has_favorite = df_funnel['first_favorite_at'].notna()
has_chat = df_funnel['first_message_at'].notna()
has_revisit = df_funnel['post_favorite_revisit_count'] > 0
favorite_before_chat = (
    has_favorite
    & has_chat
    & (df_funnel['first_favorite_at'] <= df_funnel['first_message_at'])
)

df_funnel['journey_group'] = np.select(
    [
        ~has_favorite & ~has_chat & (df_funnel['user_view_count'] == 1),
        ~has_favorite & ~has_chat & (df_funnel['user_view_count'] >= 2),
        has_favorite & ~has_chat & ~has_revisit,
        has_favorite & ~has_chat & has_revisit,
        favorite_before_chat & ~has_revisit,
        favorite_before_chat & has_revisit,
        has_chat,
    ],
    [
        'view_only',
        'repeat_view_only',
        'favorite_stalled_no_revisit',
        'favorite_stalled_revisit',
        'favorite_to_chat_no_revisit',
        'favorite_revisit_to_chat',
        'direct_chat',
    ],
    default='unclassified',
)

print('df_funnel:', df_funnel.shape)


df_funnel: (36360, 25)


In [8]:
# 퍼널 데이터 검증하기
funnel_quality = pd.Series({
    '전체 사용자-판매글 조합': len(df_funnel),
    '중복 사용자-판매글 조합': df_funnel.duplicated(['user_id', 'listing_id']).sum(),
    '미분류 행동 경로': df_funnel['journey_group'].eq('unclassified').sum(),
    '판매글 특성 결측': df_funnel['category'].isna().sum(),
})

display(funnel_quality.to_frame('건수'))


,건수
전체 사용자-판매글 조합,36360
중복 사용자-판매글 조합,0
미분류 행동 경로,0
판매글 특성 결측,0


In [9]:
# 행동 경로별 규모 계산하기
journey_order = [
    'view_only',
    'repeat_view_only',
    'favorite_stalled_no_revisit',
    'favorite_stalled_revisit',
    'favorite_to_chat_no_revisit',
    'favorite_revisit_to_chat',
    'direct_chat',
]

journey_labels = {
    'view_only': '조회만',
    'repeat_view_only': '조회 → 재조회',
    'favorite_stalled_no_revisit': '조회 → 찜 → 채팅 없음',
    'favorite_stalled_revisit': '조회 → 찜 → 재조회 → 채팅 없음',
    'favorite_to_chat_no_revisit': '조회 → 찜 → 채팅',
    'favorite_revisit_to_chat': '조회 → 찜 → 재조회 → 채팅',
    'direct_chat': '조회 → 바로 채팅',
}

journey_table = (
    df_funnel['journey_group']
    .value_counts()
    .reindex(journey_order)
    .rename_axis('journey_group')
    .reset_index(name='건수')
)

journey_table['행동 경로'] = journey_table['journey_group'].map(journey_labels)
journey_table['전체 비율(%)'] = journey_table['건수'] / len(df_funnel) * 100

display(journey_table[['행동 경로', '건수', '전체 비율(%)']].round(2))


,행동 경로,건수,전체 비율(%)
0,조회만,25209,69.3300
1,조회 → 재조회,3872,10.6500
2,조회 → 찜 → 채팅 없음,3927,10.8000
3,조회 → 찜 → 재조회 → 채팅 없음,732,2.0100
4,조회 → 찜 → 채팅,709,1.9500
5,조회 → 찜 → 재조회 → 채팅,142,0.3900
6,조회 → 바로 채팅,1769,4.8700


**중간 인사이트 1**

전체 36,360건 중 직접 채팅은 1,769건, 찜 후 채팅은 851건이다.
채팅은 찜을 반드시 거쳐야 하는 단일 퍼널이 아니라, 직접 채팅과 찜 후 채팅으로 갈라지는 분기형 행동이다.


In [10]:
# 채팅 경로별 비중 계산하기
chat_path_table = pd.DataFrame({
    '구분': ['직접 채팅', '찜 후 채팅'],
    '채팅 수': [
        df_funnel['journey_group'].eq('direct_chat').sum(),
        df_funnel['journey_group'].isin([
            'favorite_to_chat_no_revisit',
            'favorite_revisit_to_chat',
        ]).sum(),
    ],
})

chat_path_table['전체 채팅 비율(%)'] = (
    chat_path_table['채팅 수'] / chat_path_table['채팅 수'].sum() * 100
)

display(chat_path_table.round(2))


,구분,채팅 수,전체 채팅 비율(%)
0,직접 채팅,1769,67.5200
1,찜 후 채팅,851,32.4800


전체 채팅 2,620건 중 67.52%는 찜 없이 바로 시작됐고, 찜 후 채팅은 32.48%였다.


In [11]:
# 채팅별 거래 결과 불러오기
chat_quality_query = '''
WITH first_favorite AS (
    SELECT
        user_id,
        listing_id,
        MIN(favorited_at) AS first_favorite_at
    FROM favorites
    GROUP BY user_id, listing_id
)
SELECT
    c.chatroom_id,
    CASE
        WHEN f.first_favorite_at IS NOT NULL
         AND f.first_favorite_at <= c.first_message_at
        THEN '찜 후 채팅'
        ELSE '직접 채팅'
    END AS chat_path,
    t.transaction_id,
    t.transaction_status
FROM chatrooms c
LEFT JOIN first_favorite f
    ON f.user_id = c.buyer_id
   AND f.listing_id = c.listing_id
LEFT JOIN transactions t
    ON t.chatroom_id = c.chatroom_id
WHERE c.first_message_at IS NOT NULL
'''

df_chat_quality = pd.read_sql(chat_quality_query, con=engine)
print('df_chat_quality:', df_chat_quality.shape)


df_chat_quality: (2620, 4)


In [12]:
# 거래 진입·완료율 계산하기
chat_quality_table = (
    df_chat_quality
    .groupby('chat_path')
    .agg(
        채팅_수=('chatroom_id', 'size'),
        거래_단계_진입=('transaction_id', 'count'),
        거래_완료=('transaction_status', lambda x: x.eq('거래완료').sum()),
    )
    .reindex(['직접 채팅', '찜 후 채팅'])
)

chat_quality_table['거래 단계 진입률(%)'] = (
    chat_quality_table['거래_단계_진입'] / chat_quality_table['채팅_수'] * 100
)
chat_quality_table['채팅→거래 완료율(%)'] = (
    chat_quality_table['거래_완료'] / chat_quality_table['채팅_수'] * 100
)

total_quality = pd.DataFrame({
    '채팅_수': [chat_quality_table['채팅_수'].sum()],
    '거래_단계_진입': [chat_quality_table['거래_단계_진입'].sum()],
    '거래_완료': [chat_quality_table['거래_완료'].sum()],
    '거래 단계 진입률(%)': [
        chat_quality_table['거래_단계_진입'].sum() / chat_quality_table['채팅_수'].sum() * 100
    ],
    '채팅→거래 완료율(%)': [
        chat_quality_table['거래_완료'].sum() / chat_quality_table['채팅_수'].sum() * 100
    ],
}, index=['전체'])

chat_quality_table = pd.concat([chat_quality_table, total_quality])
display(chat_quality_table.round(2))


,채팅_수,거래_단계_진입,거래_완료,거래 단계 진입률(%),채팅→거래 완료율(%)
직접 채팅,1769,475,194,26.8500,10.9700
찜 후 채팅,851,224,105,26.3200,12.3400
전체,2620,699,299,26.6800,11.4100


**중간 인사이트 2 - 직접 채팅은 질이 낮은 채팅이 아니다**

- 직접 채팅의 거래 단계 진입률: 26.85%
- 찜 후 채팅의 거래 단계 진입률: 26.32%
- 전체 완료 거래 299건 중 직접 채팅 경로: 194건(64.9%)

두 경로의 거래 단계 진입률은 거의 동일했다. 직접 채팅은 단순 문의가 아니라 주요 거래 전환 경로다.


In [13]:
# 채팅 경로별 특성 비교하기
df_chat_features = df_funnel[df_funnel['first_message_at'].notna()].copy()
df_chat_features['chat_path'] = np.where(
    df_chat_features['journey_group'].eq('direct_chat'),
    '직접 채팅',
    '찜 후 채팅',
)

category_median_price = df_listing.groupby('category')['current_price'].median()
df_chat_features['relative_price'] = (
    df_chat_features['current_price']
    / df_chat_features['category'].map(category_median_price)
)

chat_feature_table = (
    df_chat_features
    .groupby('chat_path')
    .agg(
        가격_중앙값=('current_price', 'median'),
        카테고리_상대가격=('relative_price', 'median'),
        평균_사진_수=('photo_count', 'mean'),
        평균_설명_길이=('description_length', 'mean'),
        가격_제안_가능=('is_negotiable', 'mean'),
        판매자_매너온도=('seller_manner_score', 'mean'),
        동일_지역=('same_region', 'mean'),
        반복_조회=('is_repeat_viewer', 'mean'),
    )
    .reindex(['직접 채팅', '찜 후 채팅'])
)

rate_columns = ['가격_제안_가능', '동일_지역', '반복_조회']
chat_feature_table[rate_columns] = chat_feature_table[rate_columns] * 100
display(chat_feature_table.round(2))


,가격_중앙값,카테고리_상대가격,평균_사진_수,평균_설명_길이,가격_제안_가능,판매자_매너온도,동일_지역,반복_조회
chat_path,,,,,,,,
직접 채팅,"48,700.0000",1.0100,4.9500,97.7500,50.8200,38.2500,42.6200,17.3000
찜 후 채팅,"48,700.0000",1.0200,4.8600,97.4000,52.1700,38.0000,45.5900,18.3300


직접 채팅과 찜 후 채팅은 가격, 사진 수, 설명 길이, 가격 제안 가능 여부, 매너온도 등에서 큰 차이가 없었다.
정적 판매글 조건보다 사용자가 문의를 시작하기 쉬운 경로 자체가 더 중요할 가능성을 보여준다.


## 2. 찜 후 재조회 사용자의 채팅 전환 비교


In [14]:
# 찜 사용자 데이터 만들기
favorite_groups = [
    'favorite_stalled_no_revisit',
    'favorite_stalled_revisit',
    'favorite_to_chat_no_revisit',
    'favorite_revisit_to_chat',
]

df_favorite = df_funnel[df_funnel['journey_group'].isin(favorite_groups)].copy()
df_favorite['chat_converted'] = df_favorite['first_message_at'].notna().astype(int)
df_favorite['revisited_after_favorite'] = (
    df_favorite['post_favorite_revisit_count'] > 0
).astype(int)

print('df_favorite:', df_favorite.shape)


df_favorite: (5510, 27)


In [15]:
# 찜 후 채팅 전환율 계산하기
favorite_conversion_table = (
    df_favorite['chat_converted']
    .value_counts()
    .reindex([1, 0])
    .rename(index={1: '찜 후 채팅 전환', 0: '찜 후 채팅 미전환'})
    .rename_axis('결과')
    .reset_index(name='건수')
)

favorite_conversion_table['비율(%)'] = (
    favorite_conversion_table['건수'] / len(df_favorite) * 100
)

display(favorite_conversion_table.round(2))


,결과,건수,비율(%)
0,찜 후 채팅 전환,851,15.4400
1,찜 후 채팅 미전환,4659,84.5600


찜 사용자 5,510건 중 채팅 전환은 851건(15.44%), 미전환은 4,659건(84.56%)이었다.


In [16]:
# 미전환자의 채팅방 상태 확인하기
df_stalled = df_favorite[df_favorite['chat_converted'].eq(0)].copy()
df_stalled['chatroom_state'] = np.where(
    df_stalled['chatroom_created_at'].isna(),
    '채팅방 없음',
    '채팅방 생성, 첫 메시지 없음',
)

stalled_chatroom_table = (
    df_stalled['chatroom_state']
    .value_counts()
    .rename_axis('상태')
    .reset_index(name='건수')
)
stalled_chatroom_table['비율(%)'] = stalled_chatroom_table['건수'] / len(df_stalled) * 100

display(stalled_chatroom_table.round(2))


,상태,건수,비율(%)
0,채팅방 없음,4628,99.3300
1,"채팅방 생성, 첫 메시지 없음",31,0.6700


채팅 미전환 4,659건 중 99.33%는 채팅방 자체가 없었다. 이탈의 핵심 지점은 첫 메시지 이후가 아니라 채팅 시작 이전이다.


In [17]:
# 미전환자의 재조회 여부 계산하기
stalled_revisit_table = (
    df_stalled['revisited_after_favorite']
    .value_counts()
    .reindex([0, 1])
    .rename(index={0: '찜 후 재조회 없음', 1: '찜 후 재조회 있음'})
    .rename_axis('행동')
    .reset_index(name='건수')
)
stalled_revisit_table['비율(%)'] = stalled_revisit_table['건수'] / len(df_stalled) * 100

display(stalled_revisit_table.round(2))


,행동,건수,비율(%)
0,찜 후 재조회 없음,3927,84.2900
1,찜 후 재조회 있음,732,15.7100


In [18]:
# 재조회 사용자 데이터 만들기
df_revisit = df_favorite[df_favorite['revisited_after_favorite'].eq(1)].copy()
df_revisit['outcome'] = np.where(
    df_revisit['chat_converted'].eq(1),
    '재조회 후 채팅',
    '재조회 후 채팅 없음',
)

print('df_revisit:', df_revisit.shape)


df_revisit: (874, 28)


In [19]:
# 재조회 후 전환율 계산하기
revisit_outcome_table = (
    df_revisit['outcome']
    .value_counts()
    .reindex(['재조회 후 채팅', '재조회 후 채팅 없음'])
    .rename_axis('결과')
    .reset_index(name='건수')
)
revisit_outcome_table['비율(%)'] = revisit_outcome_table['건수'] / len(df_revisit) * 100

display(revisit_outcome_table.round(2))


,결과,건수,비율(%)
0,재조회 후 채팅,142,16.2500
1,재조회 후 채팅 없음,732,83.7500


**중간 인사이트 3 - 재조회는 관심 신호지만 전환 보장은 아니다**

찜 후 재조회 874건 중 채팅 전환은 142건(16.25%), 미전환은 732건(83.75%)이었다.
재조회까지 한 고관심 사용자도 대부분 채팅 직전 단계에서 정체됐다.


In [20]:
# 재조회 횟수 분포 비교하기
df_revisit['revisit_bucket'] = pd.cut(
    df_revisit['post_favorite_revisit_count'],
    bins=[0, 1, 2, np.inf],
    labels=['1회', '2회', '3회 이상'],
)

revisit_count_base = (
    df_revisit
    .groupby(['revisit_bucket', 'outcome'], observed=False)
    .size()
    .rename('건수')
    .reset_index()
)
revisit_count_base['비율(%)'] = (
    revisit_count_base['건수']
    / revisit_count_base.groupby('outcome')['건수'].transform('sum')
    * 100
)

revisit_count_table = revisit_count_base.pivot(
    index='revisit_bucket',
    columns='outcome',
    values=['건수', '비율(%)'],
)

display(revisit_count_table.round(2))


건수                비율(%)            
outcome        재조회 후 채팅 재조회 후 채팅 없음 재조회 후 채팅 재조회 후 채팅 없음
revisit_bucket                                          
1회             118.0000    610.0000  83.1000     83.3300
2회              15.0000     85.0000  10.5600     11.6100
3회 이상            9.0000     37.0000   6.3400      5.0500

채팅 전환과 미전환 그룹 모두 약 83%가 한 번만 재조회했다. 재조회 횟수 자체는 두 그룹을 구분하지 못했다.


In [21]:
# 정체 사용자의 관심 상태 확인하기
df_revisit_stalled = df_revisit[df_revisit['chat_converted'].eq(0)]

stalled_interest_table = pd.DataFrame({
    '지표': [
        '현재도 찜 유지',
        '한 번 이상 찜 취소',
        '채팅방 생성 후 첫 메시지 없음',
    ],
    '건수': [
        df_revisit_stalled['is_currently_favorited'].eq(1).sum(),
        df_revisit_stalled['favorite_remove_count'].gt(0).sum(),
        df_revisit_stalled['chatroom_created_at'].notna().sum(),
    ],
})
stalled_interest_table['비율(%)'] = (
    stalled_interest_table['건수'] / len(df_revisit_stalled) * 100
)

display(stalled_interest_table.round(2))


,지표,건수,비율(%)
0,현재도 찜 유지,549,75.0000
1,한 번 이상 찜 취소,185,25.2700
2,채팅방 생성 후 첫 메시지 없음,7,0.9600


정체 사용자 732건 중 75%는 여전히 찜을 유지했다. 관심을 완전히 잃은 사용자보다 구매 결정을 보류한 잠재 관심 사용자로 해석할 수 있다.


In [22]:
# 가격 제안별 전환율 계산하기
negotiable_table = (
    df_revisit
    .groupby('is_negotiable')
    .agg(
        전체=('user_id', 'size'),
        채팅=('chat_converted', 'sum'),
        채팅_전환율=('chat_converted', 'mean'),
    )
    .rename(index={0: '가격 제안 불가능', 1: '가격 제안 가능'})
)
negotiable_table['채팅_전환율'] = negotiable_table['채팅_전환율'] * 100

display(negotiable_table.round(2))


,전체,채팅,채팅_전환율
is_negotiable,,,
가격 제안 불가능,446,60,13.4500
가격 제안 가능,428,82,19.1600


**중간 인사이트 4 - 가격 제안 가능 여부**

재조회 사용자 중 가격 제안이 불가능한 판매글의 전환율은 13.45%, 가능한 판매글은 19.16%였다.
차이는 5.71%p로, 재조회 이후의 망설임을 줄일 수 있는 실행 가능한 조건으로 나타났다.


In [23]:
# 재조회 후 채팅 O·X 비교하기
revisit_feature_table = (
    df_revisit
    .groupby('outcome')
    .agg(
        평균_사진_수=('photo_count', 'mean'),
        평균_설명_길이=('description_length', 'mean'),
        판매자_매너온도=('seller_manner_score', 'mean'),
        동일_지역=('same_region', 'mean'),
    )
    .reindex(['재조회 후 채팅 없음', '재조회 후 채팅'])
)
revisit_feature_table['동일_지역'] = revisit_feature_table['동일_지역'] * 100

display(revisit_feature_table.round(2))


,평균_사진_수,평균_설명_길이,판매자_매너온도,동일_지역
outcome,,,,
재조회 후 채팅 없음,5.0100,101.7400,38.2700,66.6700
재조회 후 채팅,5.0400,98.2200,38.4200,65.4900


사진 수, 설명 길이, 판매자 매너온도, 동일 지역 여부는 두 그룹 간 차이가 거의 없었다.


## 3. 찜 후 채팅 전환 예측 모델

최종 모델은 미래 정보가 섞이지 않도록 최초 찜 시점 이전에 확인 가능한 네 변수만 사용한다.

1. 찜 전 반복 조회 횟수
2. 동일 지역 여부
3. 찜 전 가격 인하 여부
4. 가격 제안 가능 여부


In [24]:
# 모델 데이터 만들기
ml_query = '''
WITH ranked_favorite AS (
    SELECT
        f.*,
        ROW_NUMBER() OVER (
            PARTITION BY f.user_id, f.listing_id
            ORDER BY f.favorited_at, f.favorite_id
        ) AS rn
    FROM favorites f
),
first_favorite AS (
    SELECT
        user_id,
        listing_id,
        favorited_at AS first_favorite_at
    FROM ranked_favorite
    WHERE rn = 1
),
first_chat AS (
    SELECT
        buyer_id AS user_id,
        listing_id,
        MIN(first_message_at) AS first_message_at
    FROM chatrooms
    GROUP BY buyer_id, listing_id
),
event_features AS (
    SELECT
        f.user_id,
        f.listing_id,
        f.first_favorite_at,
        c.first_message_at,
        l.is_negotiable,
        (u.region_id = l.region_id) AS same_region,
        (
            SELECT COUNT(*)
            FROM product_views pv
            WHERE pv.user_id = f.user_id
              AND pv.listing_id = f.listing_id
              AND pv.viewed_at <= f.first_favorite_at
        ) AS view_count_before_favorite,
        (
            SELECT ph.price
            FROM price_history ph
            WHERE ph.listing_id = f.listing_id
              AND ph.record_type = 'initial'
            ORDER BY ph.recorded_at
            LIMIT 1
        ) AS initial_price,
        (
            SELECT ph.price
            FROM price_history ph
            WHERE ph.listing_id = f.listing_id
              AND ph.recorded_at <= f.first_favorite_at
            ORDER BY ph.recorded_at DESC
            LIMIT 1
        ) AS price_at_favorite
    FROM first_favorite f
    JOIN listings l
        ON l.listing_id = f.listing_id
    JOIN users u
        ON u.user_id = f.user_id
    LEFT JOIN first_chat c
        ON c.user_id = f.user_id
       AND c.listing_id = f.listing_id
)
SELECT
    user_id,
    listing_id,
    GREATEST(view_count_before_favorite - 1, 0)
        AS repeat_view_count_before_favorite,
    same_region,
    is_negotiable,
    (price_at_favorite < initial_price)
        AS price_decreased_before_favorite,
    (first_message_at IS NOT NULL)
        AS chat_converted
FROM event_features
WHERE first_message_at IS NULL
   OR first_message_at >= first_favorite_at
ORDER BY listing_id, user_id
'''

df_ml = pd.read_sql(ml_query, con=engine)
df_ml = df_ml.sort_values(['user_id', 'listing_id']).reset_index(drop=True)

model_features = [
    'repeat_view_count_before_favorite',
    'same_region',
    'price_decreased_before_favorite',
    'is_negotiable',
]

df_ml[model_features + ['chat_converted']] = (
    df_ml[model_features + ['chat_converted']].apply(pd.to_numeric)
)

print('df_ml:', df_ml.shape)


df_ml: (5510, 7)


In [25]:
# 모델 데이터 검증하기
ml_quality = pd.DataFrame({
    '결측치': df_ml[model_features + ['chat_converted']].isna().sum(),
    '고유값 수': df_ml[model_features + ['chat_converted']].nunique(),
})

print('중복 사용자-판매글 조합:', df_ml.duplicated(['user_id', 'listing_id']).sum())
display(ml_quality)


중복 사용자-판매글 조합: 0


,결측치,고유값 수
repeat_view_count_before_favorite,0,10
same_region,0,2
price_decreased_before_favorite,0,2
is_negotiable,0,2
chat_converted,0,2


In [26]:
# 모델 타깃 분포 확인하기
target_table = (
    df_ml['chat_converted']
    .value_counts()
    .reindex([0, 1])
    .rename(index={0: '채팅 미전환', 1: '채팅 전환'})
    .rename_axis('타깃')
    .reset_index(name='건수')
)
target_table['비율(%)'] = target_table['건수'] / len(df_ml) * 100

display(target_table.round(2))


,타깃,건수,비율(%)
0,채팅 미전환,4659,84.5600
1,채팅 전환,851,15.4400


모델 데이터는 5,510건이며 채팅 전환은 851건(15.44%)이다. 클래스 불균형을 고려해 Stratified 5-Fold 교차검증과 `class_weight='balanced'`를 사용한다.


In [27]:
# 변수별 전환율 비교하기
df_ml['has_repeat_view_before_favorite'] = (
    df_ml['repeat_view_count_before_favorite'] >= 1
).astype(int)

condition_variables = [
    ('찜 전 반복 조회', 'has_repeat_view_before_favorite'),
    ('찜 전 가격 인하', 'price_decreased_before_favorite'),
    ('동일 지역', 'same_region'),
    ('가격 제안 가능', 'is_negotiable'),
]

conversion_rows = []
for label, column in condition_variables:
    grouped = df_ml.groupby(column)['chat_converted'].agg(['size', 'mean'])
    rate_without = round(grouped.loc[0, 'mean'] * 100, 2)
    rate_with = round(grouped.loc[1, 'mean'] * 100, 2)
    conversion_rows.append({
        '변수': label,
        '조건 없음 표본': int(grouped.loc[0, 'size']),
        '조건 없음 전환율(%)': rate_without,
        '조건 있음 표본': int(grouped.loc[1, 'size']),
        '조건 있음 전환율(%)': rate_with,
        '전환율 차이(%p)': rate_with - rate_without,
    })

conversion_rate_table = pd.DataFrame(conversion_rows)
display(conversion_rate_table.round(2))


,변수,조건 없음 표본,조건 없음 전환율(%),조건 있음 표본,조건 있음 전환율(%),전환율 차이(%p)
0,찜 전 반복 조회,5349,15.2600,161,21.7400,6.4800
1,찜 전 가격 인하,4666,14.9200,844,18.3600,3.4400
2,동일 지역,3276,14.1300,2234,17.3700,3.2400
3,가격 제안 가능,2833,14.3700,2677,16.5900,2.2200


네 변수 모두 채팅 전환율을 높이는 방향이었다. 반복 조회의 차이가 가장 컸지만 해당 표본은 161건으로 적어 탐색적 결과로 해석한다.


In [28]:
# 변수별 상관관계 계산하기
correlation_table = (
    df_ml[model_features + ['chat_converted']]
    .corr(method='spearman')['chat_converted']
    .drop('chat_converted')
    .sort_values(key=abs, ascending=False)
    .rename('Spearman 상관계수')
    .to_frame()
)

display(correlation_table.round(4))


,Spearman 상관계수
same_region,0.0439
price_decreased_before_favorite,0.0344
is_negotiable,0.0307
repeat_view_count_before_favorite,0.0302


상관계수는 모두 0.05 미만으로 약하다. 변수별 전환율 차이는 존재하지만 개인의 채팅 여부를 단독으로 설명할 정도는 아니다.


In [29]:
# 모델 입력값 분리하기
X = df_ml[model_features].astype(float)
y = df_ml['chat_converted'].astype(int)

print('X:', X.shape)
print('채팅 전환율:', f'{y.mean() * 100:.2f}%')


X: (5510, 4)
채팅 전환율: 15.44%


In [30]:
# 두 모델 정의하기
models = {
    '로지스틱 회귀': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(
            solver='liblinear',
            class_weight='balanced',
            max_iter=2000,
            random_state=RANDOM_STATE,
        )),
    ]),
    '랜덤 포레스트': RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=5,
        class_weight='balanced_subsample',
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}


In [31]:
# 교차검증 설정하기
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

scoring = {
    'roc_auc': 'roc_auc',
    'pr_auc': 'average_precision',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
}


In [32]:
# 두 모델 교차검증하기
cv_results = []

with warnings.catch_warnings():
    warnings.filterwarnings('ignore', category=RuntimeWarning, module='sklearn')

    for model_name, model in models.items():
        result = cross_validate(
            model,
            X,
            y,
            cv=cv,
            scoring=scoring,
            n_jobs=1,
        )

        cv_results.append({
            '모델': model_name,
            'ROC-AUC': result['test_roc_auc'].mean(),
            'PR-AUC': result['test_pr_auc'].mean(),
            'Precision': result['test_precision'].mean(),
            'Recall': result['test_recall'].mean(),
            'F1': result['test_f1'].mean(),
        })


In [33]:
# 모델 성능표 만들기
model_performance_table = pd.DataFrame(cv_results).set_index('모델')
model_performance_table.loc['기준 모델'] = [0.5, y.mean(), np.nan, np.nan, np.nan]

display(model_performance_table.round(4))


,ROC-AUC,PR-AUC,Precision,Recall,F1
모델,,,,,
로지스틱 회귀,0.5498,0.1798,0.1744,0.5028,0.2576
랜덤 포레스트,0.5444,0.1711,0.1792,0.4394,0.2517
기준 모델,0.5000,0.1544,NaN,NaN,NaN


**모델 결과**

- 로지스틱 회귀 ROC-AUC: 0.5498
- 랜덤 포레스트 ROC-AUC: 0.5444
- 두 모델 모두 무작위 기준 0.5보다 조금 높지만 개인의 채팅 여부를 정확히 예측할 수준은 아니다.

모델 성능이 낮다는 결과는 가격·지역·반복 조회 등의 관측 변수만으로 사용자의 최종 결정을 충분히 설명하기 어렵다는 점을 보여준다.


In [34]:
# MySQL 연결 종료하기
engine.dispose()
print('분석 완료')


분석 완료
